# BUSI Exploratory Data Analysis

Goals:
1. Verify dataset download and directory structure
2. Visualise sample images and masks per class
3. Check class balance and image size distribution
4. Verify bounding-box derivation from GT masks
5. Confirm BUSIDataset split sizes and stratification

In [ ]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import cv2

DATA_ROOT = Path('../../data/BUSI')
assert DATA_ROOT.exists(), f'Run scripts/download_data.sh first — {DATA_ROOT} not found'

In [ ]:
# Class balance
for cls in ['benign', 'malignant', 'normal']:
    imgs = [f for f in (DATA_ROOT / cls).glob('*.png') if '_mask' not in f.name]
    print(f'{cls:12s}: {len(imgs):4d} images')

In [ ]:
# Sample grid: 3 images per class with overlaid GT mask
fig, axes = plt.subplots(2, 3, figsize=(12, 8))

for col, cls in enumerate(['benign', 'malignant']):
    imgs = sorted(f for f in (DATA_ROOT / cls).glob('*.png') if '_mask' not in f.name)[:3]
    for row, img_path in enumerate(imgs):
        img  = cv2.cvtColor(cv2.imread(str(img_path)), cv2.COLOR_BGR2RGB)
        mask = cv2.imread(str(img_path.with_name(img_path.stem + '_mask.png')), cv2.IMREAD_GRAYSCALE)
        overlay = img.copy()
        overlay[mask > 127] = [255, 0, 0]
        blended = cv2.addWeighted(img, 0.7, overlay, 0.3, 0)
        axes[col][row].imshow(blended)
        axes[col][row].set_title(f'{cls} ({row+1})', fontsize=9)
        axes[col][row].axis('off')

plt.suptitle('BUSI sample images with GT mask overlay (red)', fontsize=12)
plt.tight_layout()
plt.savefig('../../results/figures/busi_eda_samples.png', dpi=150)
plt.show()

In [ ]:
# Dataset split sizes
import sys
sys.path.insert(0, '../../src')

from medai_medsam.data.dataset import BUSIDataset

for split in ['train', 'val', 'test']:
    ds = BUSIDataset(root=str(DATA_ROOT), split=split, image_size=256, seed=42)
    labels = [ds.samples[i]['class_name'] for i in range(len(ds))]
    from collections import Counter
    print(f'{split:6s}: {len(ds):4d} samples  {dict(Counter(labels))}')